# Quick Start: Your First RL Training Loop

**Time:** < 2 minutes to understand, ~10 minutes to run

This notebook demonstrates the core RL loop for agent training in minimal code.
You'll see how GRPO works by training a tiny agent to solve math problems.

---

## The Core Idea

1. **Sample** multiple responses to the same prompt
2. **Score** each response with a reward function
3. **Compute** relative advantage (better or worse than average?)
4. **Update** the model to favor above-average responses

That's GRPO in four lines.

In [ ]:
import numpy as np

# ============================================================
# Minimal GRPO: the entire algorithm in ~30 lines
# ============================================================

def grpo_step(prompt: str, generate_fn, reward_fn, n_samples: int = 4):
    """
    One GRPO training step.
    
    Parameters
    ----------
    prompt : str
        The input prompt / question
    generate_fn : callable
        Function that generates a response given a prompt
    reward_fn : callable
        Function that scores a (prompt, response) pair -> float
    n_samples : int
        Number of completions to generate per prompt
    
    Returns
    -------
    dict with responses, rewards, and advantages
    """
    # Step 1: Generate N different completions
    responses = [generate_fn(prompt) for _ in range(n_samples)]
    
    # Step 2: Score each response
    rewards = np.array([reward_fn(prompt, r) for r in responses])
    
    # Step 3: Compute relative advantage (normalize within group)
    mean_reward = rewards.mean()
    std_reward = rewards.std() + 1e-8  # avoid division by zero
    advantages = (rewards - mean_reward) / std_reward
    
    # Step 4: In real GRPO, we'd update model weights here.
    # Positive advantage = reinforce this behavior
    # Negative advantage = suppress this behavior
    
    return {
        "responses": responses,
        "rewards": rewards.tolist(),
        "advantages": advantages.tolist(),
        "best_response": responses[np.argmax(rewards)],
        "best_reward": float(rewards.max()),
    }

## Demo: Math Problem Solver

We simulate an agent learning to solve `a + b` problems.
The reward function checks if the answer is correct.

In [ ]:
import random

# Simulated model: starts noisy, gets better over time
noise_level = 5.0  # will decrease as "training" progresses

def generate_math_answer(prompt: str) -> str:
    """Simulated LLM: extracts numbers and adds noise to answer."""
    parts = prompt.replace("What is ", "").replace("?", "").split(" + ")
    a, b = int(parts[0]), int(parts[1])
    correct = a + b
    # Add noise (simulates imperfect model)
    noisy = correct + random.randint(int(-noise_level), int(noise_level))
    return str(noisy)

def math_reward(prompt: str, response: str) -> float:
    """Binary reward: 1.0 if correct, 0.0 otherwise."""
    parts = prompt.replace("What is ", "").replace("?", "").split(" + ")
    expected = int(parts[0]) + int(parts[1])
    try:
        return 1.0 if int(response.strip()) == expected else 0.0
    except ValueError:
        return 0.0

# Run one GRPO step
result = grpo_step(
    prompt="What is 15 + 27?",
    generate_fn=generate_math_answer,
    reward_fn=math_reward,
    n_samples=4,
)

print(f"Prompt: What is 15 + 27? (correct: 42)")
print(f"Responses:  {result['responses']}")
print(f"Rewards:    {result['rewards']}")
print(f"Advantages: {[f'{a:.2f}' for a in result['advantages']]}")
print(f"Best:       {result['best_response']} (reward={result['best_reward']})")

## Simulated Training Loop

Watch the accuracy improve as we simulate GRPO training steps.
In reality, ART handles this loop with a real model and vLLM backend.

In [ ]:
# Simulate multiple training steps
noise_level = 5.0
training_log = []

prompts = [f"What is {a} + {b}?" for a, b in 
           [(15, 27), (8, 13), (42, 58), (7, 9), (33, 17)]]

for step in range(10):
    step_rewards = []
    for prompt in prompts:
        result = grpo_step(prompt, generate_math_answer, math_reward, n_samples=4)
        step_rewards.extend(result["rewards"])
    
    avg_reward = np.mean(step_rewards)
    training_log.append(avg_reward)
    
    # Simulate model improvement (reduce noise after each step)
    noise_level = max(0.0, noise_level - 0.5)
    
    print(f"Step {step+1:2d} | Avg Reward: {avg_reward:.2f} | Noise: {noise_level:.1f}")

print(f"\nAccuracy improved from {training_log[0]:.0%} to {training_log[-1]:.0%}")

## Key Takeaway

The GRPO loop is simple:
- **Generate** multiple attempts
- **Score** them
- **Reinforce** the above-average ones

No labeled data. No teacher. Just trial, error, and a reward signal.

**Next:** Module 01 dives deep into the GRPO algorithm and math.

**To use with real models:** See Module 02 (ART Framework) for connecting to vLLM and Unsloth.